In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import *

In [0]:

Tgt_Table='project_mobility.silver.taxi_clean'
src_Table='project_mobility.bronze.taxi_trips_raw'

df=spark.read.table(src_Table)

df_filtered=df.filter((df.trip_distance>0) & (df.fare_amount>0) & (df.passenger_count>0))

df_renamed=df_filtered.select(df_filtered.VendorID.alias("vendor_id") \
            ,date_trunc("second", df_filtered.lpep_pickup_datetime).alias("pickup_datetime") \
            ,date_trunc("second", df_filtered.lpep_dropoff_datetime).alias("dropoff_datetime") \
            ,df_filtered.RatecodeID.cast("int").alias("rate_code_id") \
            ,df_filtered.PULocationID.alias("pickup_location_id") \
            ,df_filtered.DOLocationID.alias("dropoff_location_id") \
            ,df_filtered.passenger_count.cast("int").alias("passenger_count") \
            ,df_filtered.trip_distance.alias("trip_distance") \
            ,df_filtered.fare_amount.alias("fare_amount") \
            ,df_filtered.extra.alias("extra_charges") \
            ,df_filtered.mta_tax.alias("mta_tax") \
            ,df_filtered.tip_amount.alias("tip_amount") \
            ,df_filtered.tolls_amount.alias("tolls_amount") \
            ,df_filtered.improvement_surcharge.alias("improvement_surcharge") \
            ,df_filtered.total_amount.alias("total_amount") \
            ,df_filtered.payment_type.cast("int").alias("payment_type") \
            ,df_filtered.trip_type.cast("int").alias("trip_type") \
            ,df_filtered.congestion_surcharge.alias("congestion_surcharge") \
            ,df_filtered.cbd_congestion_fee.alias("cbd_congestion_fee") \
            ,df_filtered.source_file.alias("source_file") \
            ,current_timestamp().alias("ingestion_timestamp") \
            ,year(df_filtered.lpep_pickup_datetime).alias("year") \
            ,month(df_filtered.lpep_pickup_datetime).alias("month")
            )


df_derived=df_renamed.withColumns({
    "pickup_date": to_date(df_renamed.pickup_datetime) \
    ,"pickup_hour": hour(df_renamed.pickup_datetime) \
    ,"day_of_week": dayofweek(df_renamed.pickup_datetime) \
    ,"day_of_week_name": date_format(df_renamed.pickup_datetime, "EEEE") \
    ,"trip_duration_minutes":round((unix_timestamp(df_renamed.dropoff_datetime)-unix_timestamp(df_renamed.pickup_datetime))/60,2) \
    ,"fare_per_mile": round(df_renamed.fare_amount/df_renamed.trip_distance,2) \
    ,"tip_percentage": round(df_renamed.tip_amount/df_renamed.fare_amount*100,2) \
    ,"is_tipped" : when(df_renamed.tip_amount>0,True).otherwise(False) \
    ,"payment_type_desc": when(df_renamed.payment_type == 1, "Credit Card")
                          .when(df_renamed.payment_type == 2, "Cash")
                          .when(df_renamed.payment_type == 3, "No Charge")
                          .when(df_renamed.payment_type == 4, "Dispute")
                          .otherwise("Unknown") \
    ,"trip_type_desc":   when(df_renamed.trip_type == 1, "Street-hail")
                          .when(df_renamed.trip_type == 2, "Dispatch")
                          .otherwise("Unknown") \
    ,"rate_code_desc":   when(df_renamed.rate_code_id == 1, "Standard")
                          .when(df_renamed.rate_code_id == 2, "JFK")
                          .when(df_renamed.rate_code_id == 3, "Newark")
                          .when(df_renamed.rate_code_id == 4, "Nassau/Westchester")
                          .when(df_renamed.rate_code_id == 5, "Negotiated")
                          .when(df_renamed.rate_code_id == 6, "Group Ride")
                          .otherwise("Unknown")
})

df_final=df_derived.withColumns({
    "is_weekend": when(df_derived.day_of_week.isin([1,7]), True).otherwise(False) \
    ,"is_rush_hour": when((df_derived.pickup_hour.isin([7,8,9,10,16,17,18,19])), True).otherwise(False) \
    ,"speed_mph": when((df_derived.trip_distance > 0) & (df_derived.trip_duration_minutes > 0),round(df_derived.trip_distance / df_derived.trip_duration_minutes * 60, 2)).otherwise(0)
})

df_final = df_final.filter((df_final.trip_duration_minutes>0) & (df_final.pickup_location_id.isNotNull()) & (df_final.dropoff_location_id.isNotNull()))

#display(df_final)

df_final.write.mode("overwrite").saveAsTable(Tgt_Table)
                               
